In [ ]:
import pandas as pd

factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_partsupp(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/partsupp.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
partsupp = load_partsupp(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# Filter supplier keys with complaints
bad_supp = supplier.loc[
    supplier["S_COMMENT"].str.contains(r"Customer.*Complaints", regex=True, na=False),
    "S_SUPPKEY"
].unique()

# Build and aggregate total in one pipeline
total = (
    part
    # filter part rows and select needed cols
    .loc[
        (part.P_BRAND != "Brand#45")
        & ~part.P_TYPE.str.startswith("MEDIUM POLISHED", na=False)
        & part.P_SIZE.isin([49, 14, 23, 45, 19, 3, 36, 9]),
        ["P_PARTKEY", "P_BRAND", "P_TYPE", "P_SIZE"]
    ]
    # join to partsupp
    .merge(
        partsupp[["PS_PARTKEY", "PS_SUPPKEY"]],
        left_on="P_PARTKEY", right_on="PS_PARTKEY",
        how="inner"
    )
    # exclude bad suppliers and drop join keys
    .loc[
        lambda df: ~df.PS_SUPPKEY.isin(bad_supp),
        ["P_BRAND", "P_TYPE", "P_SIZE", "PS_SUPPKEY"]
    ]
    # count unique suppliers per group
    .groupby(["P_BRAND", "P_TYPE", "P_SIZE"], as_index=False)
    .agg(SUPPLIER_CNT=("PS_SUPPKEY", "nunique"))
    # sort by count desc, then by brand/type/size asc
    .sort_values(
        by=["SUPPLIER_CNT", "P_BRAND", "P_TYPE", "P_SIZE"],
        ascending=[False, True, True, True]
    )
)